# Eksperimen 3 — APTOS balanced dan DDR FP32 Math recovery

**Pertanyaan penelitian:** bagaimana hasil kandidat dengan oversampling penuh hanya pada train, dan apa keterbatasan kampanye DDR yang memerlukan diagnostic recovery? Dua dataset ini membentuk generasi eksperimen yang sama tetapi **tidak memakai protokol training yang identik**. [Ringkasan utama](../../CAD_ViT.ipynb) · [V2](02_experiment_bg10_cadvit_v2.ipynb).

## Identitas dan protokol dataset

| Komponen | APTOS2019 | DDR |
| --- | --- | --- |
| Campaign/run | `balanced_v1/aptos_r16` | `cadvit_tables_1_5_balanced_fp32_math_mb8_diag/ddr_full` |
| Kelas / train unik | Grade 0–4 / 2.930 gambar | Grade 0–4 / 10.018 gambar |
| Train draw setelah oversampling | 1.445 × 5 = 7.225 per epoch | 5.014 × 5 = 25.070 per epoch |
| Manifest SHA-256 tercatat | `73d1546e3c2fd2bf9e3d0fd10af914cb523d28e4bc8e1f25212af454d4be8854` | `7bc3e3916e012bc46fcd9cd69aa469e0daf3229e1273f1d8ecb6054899bf6eb1` |
| Epoch / precision | 80; AMP FP16 | 80; FP32; recovery epoch 43–80 |
| Micro-batch × akumulasi | 16 × 1 | 16 × 1 epoch 1–42; 8 × 2 epoch 43–80 |
| Best validation epoch | 24 | 69 |

Kedua config mencatat seed 42, LR 0,000186, AdamW weight decay 0, CrossEntropyLoss, grad clip norm 1, input 512×512, augmentasi train, tanpa BG10, seleksi checkpoint validation macro-F1 dengan tie validation loss. Jumlah test: 366 APTOS, 1.252 DDR; isi manifest/val membership dan checkpoint biner tidak tersedia lokal. Draw oversampling mengulang gambar minoritas, tidak menciptakan gambar unik. Protokol ini adalah extension dari paper. [Config APTOS](../../results/APTOS2019/full_oversampling_amp_fp16_80epochs/config.json) · [config DDR](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/config.json) · [fork provenance DDR](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/FORK_PROVENANCE.json).

## Walkthrough kode dan perbedaan dari V2

| Langkah | Source | Input → output dan status |
| --- | --- | --- |
| Model | [`CADViTCandidate` di `model.py`](../cadvit/model.py) | Tensor RGB → lima logit; hash source config kedua dataset `6aff75039dbd2d3e545e6ff1998674221b3671df7d58362792e0b2ca249a0c25`. |
| Preprocessing | [`preprocess_fundus` di `preprocessing.py`](../cadvit/preprocessing.py) | Gambar PIL → array gambar RGB 512×512 hasil crop, resize, dan Gaussian blur jalur V1; **tanpa BG10**. Fungsi ini tidak menghasilkan tensor. Hash source `34d4e39bda7a3a996cc1662bfc7144c09667071a52435050491af531e1fbb7b5`. |
| Dataset dan tensor | [`dataset_for` dan kelas lokal `FundusSplit` di `experiments.py`](../cadvit/experiments.py) | `FundusSplit.__getitem__` membuka gambar, memanggil `preprocess_fundus`, menerapkan horizontal/vertical flip dan `RandomAffine` hanya bila split train memakai augmentasi, lalu mengonversi array ke tensor PyTorch `C×H×W` float32 berskala 0–1; hasilnya pasangan `(tensor, label)`. |
| Sampling/loading | [`balanced_train_indices`, `split_loader` di `experiments.py`](../cadvit/experiments.py) | Manifest train → indeks oversampling dan batch; validasi/test tidak dioversample. |
| Training | `Protocol`, `train_epoch`, `train` di modul yang sama | Config/loader → history dan checkpoint menurut validation. Hash lokal cocok dengan APTOS config, **tidak cocok** dengan runner SHA DDR recovery yang tercatat. |
| Agregasi hasil | `score`, `support_weighted_metrics` di modul; [evaluator DDR terarsip](../provenance/ddr_final_test/README.md) | Prediksi beku → metrik agregat. Source evaluator DDR yang diarsipkan terkait run test, tetapi hash source saja tidak membuktikan eksekusi. |

Jangan memakai adapter draft `ddr_recovery_final_test.py` sebagai evaluator yang dieksekusi. Jalur evaluasi akhir tidak dijalankan dari notebook ini. Source runner DDR tepat saat server recovery belum tersedia lokal, sehingga kesetaraan implementasi seluruh training DDR tidak diklaim.

## Cara membaca hasil

Semua metrik tabel epoch adalah **training/validation**. Checkpoint dipilih dengan validation macro-F1; tabel final test di bawah memakai test beku setelah pilihan itu. Matriks kebingungan memakai **baris label sebenarnya dan kolom prediksi**. Nilai final test tidak dipakai untuk memilih checkpoint. Seluruh tabel dan gambar adalah Markdown/artefak tersimpan; membuka notebook ini tidak menjalankan training atau inferensi. Angka paper dan angka reproduksi tidak otomatis memakai split atau definisi agregasi F1 yang sama.

### APTOS2019 `balanced_v1/aptos_r16` — 80 epoch

[Sumber history](../../results/APTOS2019/full_oversampling_amp_fp16_80epochs/history.jsonl), SHA-256 `7ce8490772e1c37967c7418a52b2391fce1f6636ad6d798e52e6bb29150ccdcf`. Best validation macro-F1 **0.6614213564213565** pada epoch **24**. Kurva SVG di bawah diturunkan hanya dari history yang sama, tanpa training ulang.

| Epoch | Train Loss | Validation Loss | Validation Accuracy | Validation Macro-F1 |
| ---: | ---: | ---: | ---: | ---: |
| 1 | 1.300262275 | 1.044934996 | 0.554644809 | 0.362002075 |
| 2 | 1.136483329 | 0.893349033 | 0.655737705 | 0.488097431 |
| 3 | 1.003350817 | 0.826451257 | 0.685792350 | 0.535250621 |
| 4 | 0.914153505 | 0.736215908 | 0.724043716 | 0.571386384 |
| 5 | 0.867897571 | 0.635215503 | 0.745901639 | 0.585542487 |
| 6 | 0.829912540 | 0.699582112 | 0.724043716 | 0.548545595 |
| 7 | 0.831879950 | 0.732516791 | 0.707650273 | 0.559661872 |
| 8 | 0.807091384 | 0.696549871 | 0.737704918 | 0.587705528 |
| 9 | 0.795766550 | 0.712637175 | 0.721311475 | 0.591521921 |
| 10 | 0.823810876 | 0.649130540 | 0.751366120 | 0.624572337 |
| 11 | 0.792641000 | 0.751202229 | 0.713114754 | 0.577371405 |
| 12 | 0.747555507 | 0.690401999 | 0.707650273 | 0.583081240 |
| 13 | 0.759120356 | 0.781316025 | 0.683060109 | 0.560760851 |
| 14 | 0.744483490 | 0.675960562 | 0.756830601 | 0.603069729 |
| 15 | 0.729475984 | 0.637292237 | 0.784153005 | 0.642579634 |
| 16 | 0.742072010 | 0.737431641 | 0.718579235 | 0.563450738 |
| 17 | 0.692405118 | 0.710678553 | 0.751366120 | 0.619353111 |
| 18 | 0.705312382 | 0.709754264 | 0.734972678 | 0.617879923 |
| 19 | 0.689059540 | 0.717439538 | 0.737704918 | 0.606332601 |
| 20 | 0.684146387 | 0.755579157 | 0.721311475 | 0.579415285 |
| 21 | 0.687277765 | 0.687112282 | 0.762295082 | 0.624651473 |
| 22 | 0.654615258 | 0.796989830 | 0.718579235 | 0.583938099 |
| 23 | 0.669872315 | 0.732279472 | 0.767759563 | 0.636601415 |
| **24** | 0.642142612 | 0.674392606 | 0.792349727 | 0.661421356 |
| 25 | 0.664077363 | 0.654505681 | 0.781420765 | 0.656727772 |
| 26 | 0.658552324 | 0.676959576 | 0.767759563 | 0.623383917 |
| 27 | 0.643107119 | 0.656150072 | 0.762295082 | 0.621731543 |
| 28 | 0.628942577 | 0.749212916 | 0.732240437 | 0.570349152 |
| 29 | 0.666459382 | 0.723542852 | 0.759562842 | 0.631464359 |
| 30 | 0.638240102 | 0.709785049 | 0.759562842 | 0.618390429 |
| 31 | 0.647770437 | 0.696689981 | 0.737704918 | 0.561977453 |
| 32 | 0.617989236 | 0.720002950 | 0.740437158 | 0.603769482 |
| 33 | 0.597949701 | 0.750939275 | 0.745901639 | 0.589644601 |
| 34 | 0.572920413 | 0.717154099 | 0.756830601 | 0.615568230 |
| 35 | 0.547141896 | 0.738747754 | 0.745901639 | 0.598820055 |
| 36 | 0.555442267 | 0.799295764 | 0.751366120 | 0.620235842 |
| 37 | 0.536859525 | 0.846371948 | 0.759562842 | 0.610461468 |
| 38 | 0.530475319 | 0.733561694 | 0.770491803 | 0.607041265 |
| 39 | 0.516269929 | 0.776526972 | 0.748633880 | 0.600353997 |
| 40 | 0.499785888 | 0.815584071 | 0.759562842 | 0.586108978 |
| 41 | 0.488654147 | 0.765832469 | 0.759562842 | 0.602372513 |
| 42 | 0.497761125 | 0.780162911 | 0.748633880 | 0.604180812 |
| 43 | 0.487680878 | 0.827277484 | 0.726775956 | 0.560233818 |
| 44 | 0.464704157 | 0.670799747 | 0.775956284 | 0.619953484 |
| 45 | 0.476052669 | 0.733723587 | 0.773224044 | 0.633474802 |
| 46 | 0.469695001 | 0.905130745 | 0.721311475 | 0.580118152 |
| 47 | 0.469323982 | 0.869881449 | 0.743169399 | 0.599039238 |
| 48 | 0.433401432 | 0.853997005 | 0.770491803 | 0.597404493 |
| 49 | 0.434872382 | 0.751238309 | 0.770491803 | 0.602464303 |
| 50 | 0.427021191 | 0.778591426 | 0.778688525 | 0.617778848 |
| 51 | 0.422442289 | 0.738812617 | 0.775956284 | 0.624863394 |
| 52 | 0.407805875 | 0.801775132 | 0.740437158 | 0.605539169 |
| 53 | 0.423354278 | 0.821796845 | 0.756830601 | 0.584938559 |
| 54 | 0.419648911 | 0.785132792 | 0.770491803 | 0.615659355 |
| 55 | 0.399186900 | 0.775002866 | 0.792349727 | 0.642133418 |
| 56 | 0.397283497 | 0.944822742 | 0.767759563 | 0.620803319 |
| 57 | 0.369870313 | 0.871357403 | 0.754098361 | 0.592674011 |
| 58 | 0.366528860 | 0.858351537 | 0.765027322 | 0.614404288 |
| 59 | 0.378974715 | 0.893307204 | 0.784153005 | 0.642382982 |
| 60 | 0.372492947 | 0.850394703 | 0.743169399 | 0.572085442 |
| 61 | 0.356498423 | 0.852188454 | 0.729508197 | 0.574338225 |
| 62 | 0.348418295 | 0.872985690 | 0.775956284 | 0.598961548 |
| 63 | 0.385667926 | 0.946845607 | 0.721311475 | 0.578648101 |
| 64 | 0.349643771 | 0.996219560 | 0.751366120 | 0.578105413 |
| 65 | 0.360530279 | 0.979982509 | 0.745901639 | 0.574987360 |
| 66 | 0.346584004 | 1.129620243 | 0.724043716 | 0.567427227 |
| 67 | 0.373959053 | 0.919861535 | 0.778688525 | 0.612869891 |
| 68 | 0.360797263 | 0.747373074 | 0.756830601 | 0.608913376 |
| 69 | 0.373037607 | 0.762191654 | 0.751366120 | 0.576953716 |
| 70 | 0.344657476 | 0.862721276 | 0.756830601 | 0.580989083 |
| 71 | 0.348810370 | 0.854701915 | 0.797814208 | 0.632990646 |
| 72 | 0.320258249 | 0.878561984 | 0.778688525 | 0.613172837 |
| 73 | 0.323409679 | 0.939971906 | 0.759562842 | 0.568302898 |
| 74 | 0.320448931 | 0.898562837 | 0.770491803 | 0.590485167 |
| 75 | 0.332110063 | 1.204536162 | 0.721311475 | 0.580408976 |
| 76 | 0.333395494 | 1.167203233 | 0.762295082 | 0.595723311 |
| 77 | 0.312616030 | 1.009467938 | 0.770491803 | 0.617549614 |
| 78 | 0.308761524 | 0.905323993 | 0.797814208 | 0.610958007 |
| 79 | 0.312929171 | 0.934143815 | 0.778688525 | 0.555059848 |
| 80 | 0.311636609 | 0.844685598 | 0.784153005 | 0.621923386 |

![Kurva APTOS balanced 80 epoch](../../reproduction/report_figures/aptos_balanced_80_epoch_curves.svg)

### DDR FP32 Math recovery — 80 epoch

[Sumber history](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/history.jsonl), SHA-256 `976db4b87d8854ecf8aabf0cf2228497fc0b63dbb120d83a38249681202b49fb`. Epoch **1–42** diwarisi dari run FP32 micro-batch 16; epoch **43–80** dari diagnostic recovery fork Math SDPA micro-batch 8 × accumulation 2. Best validation macro-F1 **0.6530693590977841** pada epoch **69**. Kurva PNG di bawah adalah artefak yang telah tersimpan.

| Epoch | Train Loss | Validation Loss | Validation Accuracy | Validation Macro-F1 |
| ---: | ---: | ---: | ---: | ---: |
| 1 | 1.404246424 | 1.225608672 | 0.460862620 | 0.340685089 |
| 2 | 1.133718644 | 1.109977968 | 0.500798722 | 0.387205104 |
| 3 | 1.010771972 | 1.022352522 | 0.532747604 | 0.422028868 |
| 4 | 0.965449856 | 0.968158612 | 0.583865815 | 0.442540430 |
| 5 | 0.922552022 | 1.069899990 | 0.501597444 | 0.443946121 |
| 6 | 0.882990358 | 0.888599826 | 0.644568690 | 0.507831076 |
| 7 | 0.838678385 | 1.074898594 | 0.476038339 | 0.459405136 |
| 8 | 0.801185661 | 1.011650742 | 0.544728435 | 0.502370465 |
| 9 | 0.768220572 | 0.984536150 | 0.567891374 | 0.480567873 |
| 10 | 0.741517200 | 0.873944857 | 0.634984026 | 0.524028643 |
| 11 | 0.711948839 | 0.914563011 | 0.613418530 | 0.513376692 |
| 12 | 0.725334839 | 0.964742513 | 0.584664537 | 0.507063999 |
| 13 | 0.719023004 | 0.866030887 | 0.658146965 | 0.535449245 |
| 14 | 0.680094416 | 0.813515254 | 0.653354633 | 0.568634782 |
| 15 | 0.669314694 | 0.991521537 | 0.678115016 | 0.537039046 |
| 16 | 0.661969304 | 0.984627471 | 0.580670927 | 0.513466550 |
| 17 | 0.633065336 | 0.923374027 | 0.615015974 | 0.558293386 |
| 18 | 0.595616120 | 0.920821194 | 0.656549521 | 0.560572101 |
| 19 | 0.582315808 | 0.812177504 | 0.674121406 | 0.563530948 |
| 20 | 0.569430667 | 0.775037900 | 0.706869010 | 0.556016592 |
| 21 | 0.563140095 | 0.840393492 | 0.682907348 | 0.549360967 |
| 22 | 0.577795289 | 0.897319339 | 0.642172524 | 0.532678776 |
| 23 | 0.581484702 | 0.901637542 | 0.618210863 | 0.536151746 |
| 24 | 0.557116917 | 1.529896228 | 0.549520767 | 0.417465416 |
| 25 | 0.552338988 | 0.955838982 | 0.600638978 | 0.517358834 |
| 26 | 0.525822347 | 0.852917580 | 0.667731629 | 0.560102540 |
| 27 | 0.501575095 | 0.674914998 | 0.758785942 | 0.608341094 |
| 28 | 0.491974327 | 0.793767435 | 0.698881789 | 0.577183774 |
| 29 | 0.468823797 | 0.915935204 | 0.634984026 | 0.549956453 |
| 30 | 0.473202833 | 1.042066058 | 0.594249201 | 0.523430332 |
| 31 | 0.452893406 | 0.758122848 | 0.718051118 | 0.598349290 |
| 32 | 0.428039118 | 0.723747972 | 0.742811502 | 0.620130846 |
| 33 | 0.419913940 | 0.704093206 | 0.733226837 | 0.583558076 |
| 34 | 0.410194789 | 0.880906987 | 0.683706070 | 0.575103475 |
| 35 | 0.404583828 | 0.728160021 | 0.750000000 | 0.595342656 |
| 36 | 0.391913696 | 0.859399624 | 0.683706070 | 0.566504100 |
| 37 | 0.372282817 | 0.809001746 | 0.722843450 | 0.551233565 |
| 38 | 0.381346503 | 0.720126037 | 0.761980831 | 0.611472156 |
| 39 | 0.365141683 | 0.802428166 | 0.713258786 | 0.589112809 |
| 40 | 0.355888384 | 0.775883935 | 0.745207668 | 0.589407950 |
| 41 | 0.357476182 | 0.817721587 | 0.738817891 | 0.615778871 |
| 42 | 0.325935845 | 0.803784908 | 0.741214058 | 0.612804248 |
| 43 | 0.436828966 | 0.800064616 | 0.731629393 | 0.611662989 |
| 44 | 0.414366092 | 0.711620428 | 0.763578275 | 0.612970238 |
| 45 | 0.389669588 | 0.747728577 | 0.750798722 | 0.631121148 |
| 46 | 0.382752757 | 0.786301834 | 0.745207668 | 0.598023316 |
| 47 | 0.382460871 | 0.693735215 | 0.762779553 | 0.608727392 |
| 48 | 0.360789601 | 0.699162223 | 0.769968051 | 0.629802594 |
| 49 | 0.353550779 | 0.746104798 | 0.753194888 | 0.610830263 |
| 50 | 0.352059575 | 0.708370265 | 0.773961661 | 0.627137722 |
| 51 | 0.330418980 | 0.692904777 | 0.789936102 | 0.646622508 |
| 52 | 0.337431525 | 0.667939929 | 0.785942492 | 0.621399573 |
| 53 | 0.337916337 | 0.710190548 | 0.772364217 | 0.642500881 |
| 54 | 0.323586789 | 0.714731458 | 0.789936102 | 0.647230362 |
| 55 | 0.322199898 | 1.030661828 | 0.662939297 | 0.563243400 |
| 56 | 0.321952588 | 0.944877473 | 0.719648562 | 0.611667683 |
| 57 | 0.309920754 | 0.798282763 | 0.767571885 | 0.624168551 |
| 58 | 0.302575678 | 0.668826196 | 0.797923323 | 0.632344886 |
| 59 | 0.312219306 | 0.807529964 | 0.742811502 | 0.606826804 |
| 60 | 0.305732877 | 0.931231594 | 0.707667732 | 0.592002302 |
| 61 | 0.298883509 | 0.954223416 | 0.750798722 | 0.622393540 |
| 62 | 0.286899505 | 0.814141378 | 0.739616613 | 0.609643205 |
| 63 | 0.280342935 | 0.929668438 | 0.698881789 | 0.617818874 |
| 64 | 0.291308434 | 0.724491554 | 0.786741214 | 0.634205087 |
| 65 | 0.277409195 | 0.774629542 | 0.776357827 | 0.622204287 |
| 66 | 0.279467931 | 0.940787218 | 0.753194888 | 0.614697260 |
| 67 | 0.273347682 | 0.733556145 | 0.785143770 | 0.639431770 |
| 68 | 0.271503579 | 0.877608703 | 0.750798722 | 0.641126923 |
| **69** | 0.272224328 | 0.790420573 | 0.785942492 | 0.653069359 |
| 70 | 0.272320427 | 0.808614515 | 0.765175719 | 0.628372276 |
| 71 | 0.283456829 | 0.725746319 | 0.789936102 | 0.643743127 |
| 72 | 0.263403548 | 0.724641497 | 0.785942492 | 0.620102969 |
| 73 | 0.259554148 | 0.738315662 | 0.785942492 | 0.631041419 |
| 74 | 0.246140040 | 0.763701600 | 0.785942492 | 0.647903674 |
| 75 | 0.238237324 | 0.900021472 | 0.747603834 | 0.625594977 |
| 76 | 0.253733912 | 0.768231712 | 0.791533546 | 0.633787852 |
| 77 | 0.240036404 | 0.782074357 | 0.782747604 | 0.647051170 |
| 78 | 0.235021987 | 0.844549232 | 0.755591054 | 0.617421003 |
| 79 | 0.243833820 | 0.988419307 | 0.732428115 | 0.631466952 |
| 80 | 0.245128337 | 0.833879018 | 0.788338658 | 0.626342701 |

![DDR train loss](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/train_loss.png)

![DDR validation loss](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/validation_loss.png)

![DDR validation accuracy](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/validation_accuracy.png)

![DDR validation macro-F1](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/validation_macro_f1.png)

## Final test beku — campaign ketiga

| Dataset/run | Epoch pilihan validation | Best validation macro-F1 | N test | Benar | Accuracy test | Macro-F1 test | Weighted-F1 test | Macro specificity test |
| --- | ---: | ---: | ---: | ---: | ---: | ---: | ---: | ---: |
| APTOS `balanced_v1/aptos_r16` | 24 | 0,6614213564213565 | 366 | 268 | 0,73224043715847 | 0,5315285435735081 | 0,7449461138522905 | 0,9360795785213256 |
| DDR `cadvit_tables_1_5_balanced_fp32_math_mb8_diag/ddr_full` | 69 | 0,6530693590977841 | 1.252 | 975 | 0,7787539936102237 | 0,6319812217431724 | 0,8011143014317892 | 0,9410946313470154 |

Sumber: [APTOS final-test summary](../../results/APTOS2019/full_oversampling_amp_fp16_80epochs/final_test_summary.json) dan [DDR final-test summary](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/final_test_summary.json). DDR juga melaporkan precision makro **0,6214465962662282**, recall makro **0,6840525897005094**, QWK **0,8370787904943411**, dan grade MAE **0,305111821086262**. Nilai validation DDR 65,31% dan final-test DDR 63,20% berasal dari split berbeda. Prediksi lengkap per gambar DDR tetap di server; sembilan contoh terpilih untuk APTOS dan DDR kini ditampilkan pada [notebook analisis kesalahan](../notebooks/07_error_analysis.ipynb), dengan panel APTOS hanya tersedia lokal dan diabaikan Git.

### APTOS balanced FP16 — final test

| True grade \ Predicted grade | 0 | 1 | 2 | 3 | 4 | Support |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 169 | 8 | 1 | 2 | 0 | 180 |
| 1 | 2 | 19 | 13 | 2 | 2 | 38 |
| 2 | 1 | 13 | 65 | 14 | 7 | 100 |
| 3 | 0 | 1 | 6 | 4 | 8 | 19 |
| 4 | 0 | 0 | 8 | 10 | 11 | 29 |

| Grade | Support | Precision | Recall | F1 | Specificity |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 180 | 0.982558 | 0.938889 | 0.960227 | 0.983871 |
| 1 | 38 | 0.463415 | 0.500000 | 0.481013 | 0.932927 |
| 2 | 100 | 0.698925 | 0.650000 | 0.673575 | 0.894737 |
| 3 | 19 | 0.125000 | 0.210526 | 0.156863 | 0.919308 |
| 4 | 29 | 0.392857 | 0.379310 | 0.385965 | 0.949555 |

### DDR FP32 Math recovery — final test

| True grade \ Predicted grade | 0 | 1 | 2 | 3 | 4 | Support |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 530 | 74 | 17 | 0 | 5 | 626 |
| 1 | 12 | 37 | 14 | 0 | 0 | 63 |
| 2 | 41 | 52 | 325 | 16 | 14 | 448 |
| 3 | 0 | 1 | 11 | 11 | 0 | 23 |
| 4 | 1 | 2 | 10 | 7 | 72 | 92 |

| Grade | Support | Precision | Recall | F1 | Specificity |
| ---: | ---: | ---: | ---: | ---: | ---: |
| 0 | 626 | 0.907534 | 0.846645 | 0.876033 | 0.913738 |
| 1 | 63 | 0.222892 | 0.587302 | 0.323144 | 0.891505 |
| 2 | 448 | 0.862069 | 0.725446 | 0.787879 | 0.935323 |
| 3 | 23 | 0.323529 | 0.478261 | 0.385965 | 0.981286 |
| 4 | 92 | 0.791209 | 0.782609 | 0.786885 | 0.983621 |

![Confusion matrix DDR recovery asli](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/confusion_matrix.png)

## Interpretasi dan batas kausal

APTOS balanced: F1 grade 3 **0,156863** dengan support 19; grade 2→3 muncul 14 kali. DDR recovery: grade 1 memiliki F1 terendah (**0,323144**); grade 3 hanya **11/23** benar dan grade 2→3 terjadi 16 kali, sedangkan grade 3→2 terjadi 11 kali. Ini pola agregat, bukan bukti penyebab klinis. Oversampling mengulang gambar minoritas sehingga paparan train dan jumlah unik harus dibedakan.

DDR epoch 1–42 berasal dari FP32 micro-batch 16. Setelah instabilitas numerik, epoch 43–80 memakai FP32 Math SDPA, micro-batch 8, akumulasi 2. Efektif batch 16 tidak menjadikan langkah optimizer/BatchNorm identik. Perubahan serentak membatasi atribusi hasil. APTOS AMP dan DDR FP32 juga tidak dapat disamakan sebagai satu kontrol. Validation memilih checkpoint; final test tidak menentukan run terbaik. Kedua hasil belum diuji pada populasi klinis eksternal.

## Provenance dan inspeksi aman

[History APTOS](../../results/APTOS2019/full_oversampling_amp_fp16_80epochs/history.jsonl) memiliki 80 epoch dan hash `7ce8490772e1c37967c7418a52b2391fce1f6636ad6d798e52e6bb29150ccdcf`; [history DDR](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/history.jsonl) juga 80 epoch, hash `976db4b87d8854ecf8aabf0cf2228497fc0b63dbb120d83a38249681202b49fb`. [Summary APTOS](../../results/APTOS2019/full_oversampling_amp_fp16_80epochs/summary.json) dan [summary DDR](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/summary.json) mencatat seleksi validation. [Checkpoint manifest DDR](../../results/DDR/full_oversampling_fp32_math_mb8_80epochs/checkpoint_manifest.json) mencatat `best.pt` epoch 69 SHA `84e3272d3f10d9179fd2dd9e2799b640962e1185981856f87bcb3876b9deace0`; berkas biner tidak di repo. Raw server result SHA yang dicatat: `4575f79eb536999c3ee0a9cdfb4a52dd01cf613020a1174556563df2367e1ea3`.

Hash runner APTOS `ddf5f8d55c909bb2f7c3b0ce96c125342a453539bc0ca0063c31080db6aed2b7` cocok dengan `experiments.py` lokal. Config DDR mencatat runner hash `9930ca80a8bf78ac6030bfa7d8b138fdf6fc9817f49cb8a8fd9b2d4a82e2baae`, **berbeda** dari file lokal; source tepat dan lineage recovery runner belum diaudit. [Evaluator DDR terarsip](../provenance/ddr_final_test/README.md) memiliki hash source yang cocok dengan catatan server, namun tidak membuktikan eksekusi secara kriptografis. Baca JSON/PNG secara langsung; **jangan** menjalankan ulang final-test inference.